In [1]:
import bayesflow as bf
import sys
sys.path.append("../")
from src.generative_models import *

2025-03-21 15:34:30.053520: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-21 15:34:30.053562: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-21 15:34:30.055316: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-21 15:34:30.066300: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-21 15:34:31.032175: W tensorflow/compiler/tf2

In [2]:
summary_net = bf.networks.SetTransformer(input_dim=7, summary_dim=32)

inference_net = bf.networks.InvertibleNetwork(
    num_params=3,
    coupling_settings={"dense_args": dict(kernel_regularizer=None), "dropout": False}
)

2025-03-21 15:34:34.027449: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11525 MB memory:  -> device: 0, name: NVIDIA TITAN Xp, pci bus id: 0000:04:00.0, compute capability: 6.1


## PT Model

In [3]:
pt_amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

In [4]:
pt_trainer = bf.trainers.Trainer(
    generative_model=pt_model, 
    amortizer=pt_amortizer, 
    configurator=pt_configurator, 
    checkpoint_path=f"../checkpoints/{pt_model.name}_inv_util_2",
    max_to_keep=1
)

INFO:root:Initialized empty loss history.
INFO:root:Initialized networks from scratch.
INFO:root:Performing a consistency check with provided components...
2025-03-21 15:34:40.060153: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
INFO:root:Done.


In [ ]:
history = pt_trainer.train_online(100, 1000, 128)

2025-03-21 15:35:01.021990: I external/local_xla/xla/service/service.cc:168] XLA service 0x14b8ff488d10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-03-21 15:35:01.022020: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA TITAN Xp, Compute Capability 6.1
2025-03-21 15:35:01.029115: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-03-21 15:35:01.050218: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
I0000 00:00:1742567701.151743 1232296 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Training epoch 1:  48%|████▊     | 476/1000 [01:31<00:59,  8.76it/s, Epoch: 1, Iter: 476,Loss: 2.685,Avg.Loss: 3.350,LR: 5.00E-04]

## MVL model

In [ ]:
mvl_amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

In [ ]:
trainer = bf.trainers.Trainer(
    generative_model=mvl_model, 
    amortizer=mvl_amortizer, 
    configurator=mvl_configurator, 
    checkpoint_path=f"../checkpoints/{mvl_model.name}",
    max_to_keep=1
)

In [ ]:
history = trainer.train_online(100, 1000, 128)